In [1]:
import dolfinx
from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

from Training_utils import *
from FEniCSx_PyTorch_interface import Data_to_solver, fem_solver, self_supervised_train

from dolfinx.io import XDMFFile
from mpi4py import MPI
from SPDE_problems import int_to_prblm


tset = graph_dataset(f"data/training_set_edge_attr/input_values")

class batched_loss_fn():
    def __init__(self, set):
        self.fsl = {}
        for G in set:
            num = G.mesh_id[0]
            with XDMFFile(MPI.COMM_WORLD, f"data/training_set/mesh_files/mesh_{G.mesh_id[0]}.xdmf", "r") as xdmf:
                mesh = xdmf.read_mesh(name="mesh")


            fs = int_to_prblm(idx=G.prblm_id, mesh=mesh)
            self.fsl[int(G.mesh_id)] = fem_solver(fs)

    def __call__(self, ptr, idx, y):
        loss_vals = [self.fsl[int(idx[i])](y[ptr[i]:ptr[i+1]] ) for i in range(len(ptr)-1)]
        return torch.stack(loss_vals).sum()
    
loss_fn = batched_loss_fn(tset)


class gat(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = tg.nn.models.GAT(
            in_channels=10, 
            hidden_channels=5, 
            num_layers=10, 
            out_channels=1, 
            v2=True, 
            #dropout=0., 
            act=torch.relu, 
            #norm=torch_geometric.nn.norm.LayerNorm(1),
            add_self_loops=False,
            edge_dim=4,
            residual=False
        )

    def forward(self, data) -> torch.Tensor:
        x, edge_index, edge_attr, upper = data.x, data.edge_index, data.edge_attr, data.upper
        h = self.model(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr
        )
        return upper*torch.sigmoid(h)
    
batch_size = 15
loader = train_loader(batch_size=batch_size, set=tset)
model=gat()


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.8, patience=50)
curr_loss = 10



In [6]:
batch_size = 50
loader = train_loader(batch_size=batch_size, set=tset)

In [7]:

for i in range(1000):
    loss = self_supervised_train(model=model, loader=loader,loss_fn=loss_fn, optimizer=optimizer, device='cpu')
    if curr_loss > loss:    
        print(f"iteration {i}: new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/GATv2_self_supervised_edge_attr.pth")
        #scheduler.step(loss)
    else:
        print(f"iteration {i}: {loss}")
        scheduler.step(loss)

        


iteration 0: new loss: 0.39844233602732193
iteration 1: new loss: 0.398442123686085
iteration 2: new loss: 0.39844173523997806
iteration 3: new loss: 0.3984380455092688
iteration 4: 0.39848153793744506
iteration 5: 0.3984724692273961
iteration 6: 0.3984604541495235
iteration 7: new loss: 0.39843630050923323
iteration 8: 0.398461892111279
iteration 9: new loss: 0.39841899302159206
iteration 10: 0.3984403704308141
iteration 11: 0.39845823793999363
iteration 12: new loss: 0.39841275620907174
iteration 13: new loss: 0.3984105112132077
iteration 14: 0.3984286852109087
iteration 15: 0.39848852082489183
iteration 16: 0.39842579608121426
iteration 17: new loss: 0.39840532125069994
iteration 18: new loss: 0.3983970818821738
iteration 19: 0.39843528647016185
iteration 20: 0.39840306165075906
iteration 21: new loss: 0.3983881449107891
iteration 22: 0.3983929545993512
iteration 23: 0.3983937521499518
iteration 24: 0.3984047610604607
iteration 25: 0.3983961966181162
iteration 26: new loss: 0.398378

KeyboardInterrupt: 